# L6b: Single Index Model Portfolio Allocation
In this lecture, we put the single index model (SIM) of L6a to work. L5b built minimum-variance portfolios, the efficient frontier, and the capital allocation line from two estimated inputs, the mean growth-rate vector and the growth-rate covariance matrix, with one covariance per pair of assets. L6a showed that a SIM supplies both inputs from one intercept, one beta, and one residual variance per asset plus the market's mean and variance. Today we assemble those SIM inputs for a portfolio, ask exactly how they differ from the data-driven inputs (less than you might expect, and in a way we can write down), solve the risky-only and the risky and risk-free allocation problems with them, recall the capital allocation line, the tangent portfolio, and the separation result from L5b in the form the course package actually computes, and then compare the SIM and data-driven portfolios in sample and on a year of prices neither saw.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct and diagnose SIM portfolio inputs:__ Assemble the SIM mean vector and covariance matrix from estimated intercepts, betas, residual variances, and the training-period market moments, verify that the SIM mean vector reproduces the sample means when both come from the same data, and identify the residual covariances the SIM omits.
> * __Solve SIM allocation problems:__ Compute long-only risky-asset and risky and risk-free minimum-variance portfolios from SIM inputs, recover the tangent portfolio from the ray of risky and risk-free solutions, and state where position bounds, a borrowing restriction, or a higher borrowing rate change the picture.
> * __Evaluate what the model changes:__ Compare SIM and data-driven weights and covariance regret in sample, then evaluate frozen allocations on the 2025 test period without treating one split as a validation of either input set.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Single index model inputs and minimum-variance portfolios](CHEME-5660-L6b-Example-SIM-MinVar-RA-Fall-2026.ipynb). Assemble the SIM mean vector and covariance matrix for thirteen firms from the L6a archive, verify that the SIM mean vector reproduces the sample means and that the SIM covariance is the sample covariance with the residual covariances removed, trace the long-only efficient frontier from each input set, compare the weights and the regret of the SIM portfolios under the data covariance, and score the portfolios on 2025 data.

The second example adds the risk-free asset:

> [▶ SIM portfolios with a risk-free asset, the tangent portfolio, and the capital allocation line](CHEME-5660-L6b-Example-SIM-MinVar-RRFA-Fall-2026.ipynb). Solve the risky and risk-free problem with SIM inputs as the package poses it, show that every solution lies on one ray from the risk-free asset, read the tangent portfolio off the ray and check it against the maximum-Sharpe point of the risky-only frontier, compare it with the data-driven tangent portfolio, and score complete portfolios that lend, hold, or borrow on 2025 data.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: The Single Index Model and Its Covariance
Last time we introduced the single index model. Let's quickly review the parts we need today.

> __Key Idea:__ The single index model explains each asset's growth rate by its exposure to one market index plus a residual of its own. Under its assumptions the whole covariance matrix of a portfolio follows from one beta and one residual variance per asset plus the market variance, and the mean vector from one intercept per asset plus the market mean.

For asset $i$ in a portfolio $\mathcal{P}$ of $|\mathcal{P}|$ risky assets on trading day $t$, with growth rate $g_{i,t}$ (units: inverse years), the growth rate $g_{M,t}$ of the market index $M$ (SPY in this course; the letter $M$ names the index), an intercept $\alpha_{i}$ (inverse years), a market exposure $\beta_{i}$ (dimensionless), and a residual $\varepsilon_{i,t}$ (inverse years), the model and its population assumptions are:
$$
\begin{align*}
g_{i,t} &= \alpha_{i} + \beta_{i}\,g_{M,t} + \varepsilon_{i,t},\qquad
\mathbb{E}\left[\varepsilon_{i,t}\right] = 0,\quad
\text{Cov}\left(\varepsilon_{i,t},g_{M,t}\right) = 0,\quad
\text{Cov}\left(\varepsilon_{i,t},\varepsilon_{j,t}\right) = 0\quad(i\neq j)
\end{align*}
$$
The first two assumptions make $\alpha_{i}$ and $\beta_{i}$ the least-squares projection of $g_{i}$ on $g_{M}$; the third, that residuals of different assets are __uncorrelated__, is the extra restriction that turns the model into a covariance model. Stack the assets into vectors $\mathbf{g}$, $\boldsymbol{\alpha}$, $\boldsymbol{\beta}$, and $\boldsymbol{\varepsilon}$, write $\bar{g}_{M}=\mathbb{E}[g_{M}]$ and $\sigma^{2}_{g,M}=\text{Var}(g_{M})$ for the market mean and variance, and let $\mathbf{D}_{g}=\text{diag}(\sigma^{2}_{g,\varepsilon,1},\dots,\sigma^{2}_{g,\varepsilon,|\mathcal{P}|})$ hold the residual variances $\sigma^{2}_{g,\varepsilon,i}=\text{Var}(\varepsilon_{i})$; then the mean vector and covariance matrix of the growth rates are (L6a):
$$
\boxed{
\begin{align*}
\bar{\mathbf{g}} = \mathbb{E}\left[\mathbf{g}\right] &= \boldsymbol{\alpha}+\boldsymbol{\beta}\,\bar{g}_{M}\\
\mathbf{\Sigma}_{g} = \text{Cov}\left(\mathbf{g}\right) &= \sigma^{2}_{g,M}\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top} + \mathbf{D}_{g}
\end{align*}}
$$
that is, $\text{Cov}(g_{i},g_{j})=\beta_{i}\beta_{j}\sigma^{2}_{g,M}$ for $i\neq j$ and $\text{Var}(g_{i})=\beta_{i}^{2}\sigma^{2}_{g,M}+\sigma^{2}_{g,\varepsilon,i}$: rank one plus diagonal, $2|\mathcal{P}|+1$ numbers for the covariance and $|\mathcal{P}|+1$ for the mean, in place of $|\mathcal{P}|(|\mathcal{P}|+1)/2$ covariances.

> __Derivation:__ Subtracting the mean, $g_{i}-\mathbb{E}[g_{i}]=\beta_{i}(g_{M}-\bar{g}_{M})+\varepsilon_{i}$. Multiplying the deviations of assets $i$ and $j$ and taking expectations gives four terms: $\beta_{i}\beta_{j}\,\mathbb{E}[(g_{M}-\bar{g}_{M})^{2}]=\beta_{i}\beta_{j}\sigma^{2}_{g,M}$; two cross terms $\beta_{i}\,\text{Cov}(g_{M},\varepsilon_{j})$ and $\beta_{j}\,\text{Cov}(g_{M},\varepsilon_{i})$, both zero by the second assumption; and $\text{Cov}(\varepsilon_{i},\varepsilon_{j})$, which is zero for $i\neq j$ by the third assumption and $\sigma^{2}_{g,\varepsilon,i}$ for $i=j$. $\blacksquare$

For fixed weights $\mathbf{w}$, the linearized one-period portfolio growth rate $g_{p}=\mathbf{w}^{\top}\mathbf{g}$ (L5b) has portfolio beta $\beta_{p}=\mathbf{w}^{\top}\boldsymbol{\beta}$ and variance $\text{Var}(g_{p})=\beta_{p}^{2}\sigma^{2}_{g,M}+\mathbf{w}^{\top}\mathbf{D}_{g}\mathbf{w}$, market risk plus residual risk. Everything above is a population statement. The L6a estimation example fitted the model by least squares for every security in the 2014 to 2024 data and saved the estimates in an archive; that archive is where today's inputs come from. If we did not finish that example, we look at it now.

> __Example__
>
> [▶ Estimate single index models from historical data](../L6a/CHEME-5660-L6a-Example-SVD-SIM-Estimation-Fall-2026.ipynb). Build the growth-rate matrix from the 2014 to 2024 data, fit the SIM for one firm by least squares three equivalent ways, compute its classical uncertainty and fit statistics next to those of an index fund, then fit every security in the dataset, check the residual-correlation assumption the SIM covariance relies on, and save the parameter archive that today's examples load.

___

## SIM Portfolio Inputs from the Archive
The archive holds, for each security fitted against SPY on the $N$ trading days of the training period, the least-squares estimates $\hat{\alpha}_{i}$ and $\hat{\beta}_{i}$ and the residual variance estimate $s^{2}_{g,\varepsilon,i}=\lVert\mathbf{r}_{i}\rVert_{2}^{2}/(N-2)$ (L6a; $\mathbf{r}_{i}$ is the fitted residual series of asset $i$), and, once, the sample mean $g^{\prime}_{M}$ and sample variance $s^{2}_{g,M}$ of the market growth rate over the __same__ training days. Substituting the estimates for the population quantities gives the SIM inputs for a portfolio $\mathcal{P}$:
$$
\boxed{
\begin{align*}
\hat{\mathbf{g}}_{\text{SIM}} &= \hat{\boldsymbol{\alpha}} + \hat{\boldsymbol{\beta}}\,g^{\prime}_{M}\\
\hat{\mathbf{\Sigma}}_{g,\text{SIM}} &= s^{2}_{g,M}\,\hat{\boldsymbol{\beta}}\hat{\boldsymbol{\beta}}^{\top} + \hat{\mathbf{D}}_{g},\qquad
\hat{\mathbf{D}}_{g} = \text{diag}\left(s^{2}_{g,\varepsilon,1},\dots,s^{2}_{g,\varepsilon,|\mathcal{P}|}\right)
\end{align*}}
$$
The market moments must come from the training period, the window the intercepts and betas were fitted on. Taking them from a later window (the 2025 test year, say) would mix an in-sample fit with out-of-sample information and would make every comparison below meaningless; the archive stores the training-period values so that this cannot happen by accident. All quantities are in growth-rate units: means in inverse years, variances in inverse years squared, no time-step factor anywhere (L6a).

How do these compare with the data-driven inputs of L5b, the sample-mean vector $\hat{\mathbf{g}}$ and the sample covariance $\hat{\mathbf{\Sigma}}_{g}$ of the same firms over the same days? Two facts, both exact, settle it.

> __The SIM mean vector is the sample-mean vector.__ A least-squares line fitted with an intercept passes through the point of sample means: the residuals sum to zero, so $\hat{\alpha}_{i}+\hat{\beta}_{i}\,g^{\prime}_{M}=g^{\prime}_{i}$ for every asset. Hence $\hat{\mathbf{g}}_{\text{SIM}}=\hat{\mathbf{g}}$ whenever the SIM was fitted on the same trading days used for the sample means. The two approaches __agree on the reward input__; the SIM changes only the covariance.

> __The SIM covariance is the sample covariance with the residual covariances removed.__ Least squares makes each residual series orthogonal to the column of ones and to the market series, so the sample covariance of two assets decomposes exactly as $\hat{\Sigma}_{g,ij}=\hat{\beta}_{i}\hat{\beta}_{j}\,s^{2}_{g,M}+\widehat{\text{Cov}}(\mathbf{r}_{i},\mathbf{r}_{j})$. Off the diagonal, the SIM keeps the first term and drops the second: $\hat{\mathbf{\Sigma}}_{g}-\hat{\mathbf{\Sigma}}_{g,\text{SIM}}$ is exactly the matrix of __residual covariances__, the quantities whose correlations L6a's example measured across the dataset. On the diagonal the sample variance divides the squared residuals by $N-1$ where $s^{2}_{g,\varepsilon,i}$ divides by $N-2$, a factor of $(N-1)/(N-2)$ on the residual term, about $1.0004$ for eleven years of daily data.

So the SIM covariance error is not a vague "model approximation": off the diagonal it is the residual covariance, pair by pair (and on the diagonal only the negligible degrees-of-freedom factor), and we know its sign structure from L6a. In the example's universe the residuals of firms in the same industry are positively correlated (the banks with each other by several tenths, the automakers, the semiconductor firms) and the residuals of the technology firms and the banks are negatively correlated by two or three tenths, so the SIM understates the correlation inside an industry and overstates it across those two groups. The SIM correlation matrix has a single pattern, $\hat{\rho}^{\text{SIM}}_{ij}=\hat{\beta}_{i}\hat{\beta}_{j}s^{2}_{g,M}/(\hat{\sigma}_{g,i}\hat{\sigma}_{g,j})$ for $i\neq j$ with $\hat{\sigma}_{g,i}$ the SIM's own total growth-rate standard deviation of asset $i$, so it is set by the betas and the total standard deviations and has no blocks; the data have blocks.

> __Positive definiteness:__ For any vector $\mathbf{x}$, $\mathbf{x}^{\top}\hat{\mathbf{\Sigma}}_{g,\text{SIM}}\mathbf{x}=s^{2}_{g,M}(\hat{\boldsymbol{\beta}}^{\top}\mathbf{x})^{2}+\sum_{i}s^{2}_{g,\varepsilon,i}x_{i}^{2}\geq0$, so the SIM covariance is always positive semidefinite, and it is positive definite whenever every residual variance is strictly positive (a sufficient condition; with $s^{2}_{g,M}>0$ a single zero residual variance is harmless if that asset's beta is nonzero). Positive definiteness is what the closed forms of L5b require and what makes the variance objective strictly convex; it says nothing about whether the matrix describes future co-movement well.
___

## Risky Asset Portfolios using Single Index Models
With the inputs in hand, the allocation problem is L5b's. For a portfolio $\mathcal{P}$ of risky assets with weights $\mathbf{w}$, a target growth rate $g_{\star}$ chosen by the investor, and the long-only constraint, the minimum-variance weights solve:
$$
\boxed{
\begin{align*}
\text{minimize}~&\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\text{SIM}}\mathbf{w} = \underbrace{s^{2}_{g,M}\left(\hat{\boldsymbol{\beta}}^{\top}\mathbf{w}\right)^{2}}_{\text{market risk}} + \underbrace{\sum_{i\in\mathcal{P}}s^{2}_{g,\varepsilon,i}\,w_{i}^{2}}_{\text{residual risk}}\\
\text{subject to}~&\hat{\mathbf{g}}_{\text{SIM}}^{\top}\mathbf{w}\geq g_{\star}\\
&\mathbf{1}^{\top}\mathbf{w}=1\\
&0\leq w_{i}\leq 1\qquad\forall{i}\in\mathcal{P}
\end{align*}}
$$
The objective splits into the market term, which depends on the weights only through the portfolio beta $\hat{\boldsymbol{\beta}}^{\top}\mathbf{w}$, and the residual term, a weighted sum of squares that diversification shrinks; the target enters as a floor, so sweeping $g_{\star}$ from the global minimum-variance (GMV) growth rate upward traces the GMV portfolio and the efficient branch of the frontier, exactly as in L5b, and the course package solves it with the same convex quadratic program.

Because the two input sets share the mean vector, __every difference between the SIM frontier and the data-driven frontier comes from the residual covariances__ (plus the negligible diagonal degrees-of-freedom factor). For the same weights,
$$
\begin{align*}
\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\text{SIM}}\mathbf{w} &= \mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{w} - \sum_{i\neq j}w_{i}w_{j}\,\widehat{\text{Cov}}\left(\mathbf{r}_{i},\mathbf{r}_{j}\right) + \sum_{i\in\mathcal{P}}\frac{\lVert\mathbf{r}_{i}\rVert_{2}^{2}}{(N-1)(N-2)}\,w_{i}^{2}
\end{align*}
$$
where the middle sum runs over ordered pairs and the last term is the tiny degrees-of-freedom correction. So the SIM __understates__ the variance of a portfolio concentrated in assets whose residuals are positively correlated (the semiconductor names, in the example) and __overstates__ the variance of a portfolio spread across assets whose residuals are negatively correlated (technology together with banks): what the SIM believes about its own portfolio is optimistic at one end of the frontier and pessimistic at the other. That is a statement about the SIM's own risk axis, which is why the two raw frontiers are not directly comparable. The honest comparison evaluates the SIM-optimal weights $\mathbf{w}_{\text{SIM}}(g_{\star})$ under the data covariance and compares with the data-optimal weights $\mathbf{w}(g_{\star})$ at the __same__ target:
$$
\begin{align*}
\text{regret}(g_{\star}) &= \mathbf{w}_{\text{SIM}}(g_{\star})^{\top}\hat{\mathbf{\Sigma}}_{g}\,\mathbf{w}_{\text{SIM}}(g_{\star}) - \mathbf{w}(g_{\star})^{\top}\hat{\mathbf{\Sigma}}_{g}\,\mathbf{w}(g_{\star})\geq0
\end{align*}
$$
which cannot be negative, because the data-optimal weights minimize exactly that quantity over the same feasible set. The regret is the in-sample price of the SIM's parsimony. Its square root is not a standard deviation, so the example reports it two ways: as the variance regret itself, and as the gap between the two portfolios' growth-rate standard deviations under the data covariance, which runs from a few thousandths to a tenth of a unit for its thirteen firms (largest at the concentrated high-growth end); the SIM and data-driven weights select nearly the same firms and differ by up to seven percentage points.

> __Example__
>
> [▶ Single index model inputs and minimum-variance portfolios](CHEME-5660-L6b-Example-SIM-MinVar-RA-Fall-2026.ipynb). Assemble the SIM mean vector and covariance matrix for thirteen firms from the L6a archive, verify that the SIM mean vector reproduces the sample means and that the SIM covariance is the sample covariance with the residual covariances removed, trace the long-only efficient frontier from each input set, compare the weights and the regret of the SIM portfolios under the data covariance, and score the portfolios on 2025 data.

___

## Risky and Risk-Free Asset Portfolios using Single Index Models
Now add the risk-free asset of L5b: a horizon-matched payoff with a known growth rate $g_{f}$ (units: inverse years) and zero variance over the holding period, in practice a Treasury bill or a zero-coupon Treasury (a STRIP) held to a maturity that matches the horizon; sold earlier it carries the interest-rate risk of L2b, so "risk-free" is a statement about the matched horizon, not about the security. Let $w_{f}$ be the fraction of wealth in the risk-free asset and $\mathbf{w}$ the risky weights, so that $w_{f}+\mathbf{1}^{\top}\mathbf{w}=1$. The problem the course package solves is:
$$
\boxed{
\begin{align*}
\text{minimize}~&\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\text{SIM}}\mathbf{w}\\
\text{subject to}~&\hat{\mathbf{g}}_{\text{SIM}}^{\top}\mathbf{w}+\underbrace{\left(1-\mathbf{1}^{\top}\mathbf{w}\right)}_{w_{f}}g_{f}\geq g_{\star}\\
&0\leq w_{i}\leq 1\qquad\forall{i}\in\mathcal{P}
\end{align*}}
$$
Only the risky weights carry variance, and only they are bounded. The risk-free fraction $w_{f}=1-\mathbf{1}^{\top}\mathbf{w}$ is whatever the risky weights leave over and is __not__ sign-constrained: $w_{f}<0$ means borrowing at $g_{f}$, and the package allows it, limited only by the bounds on the risky positions. Eliminating $w_{f}$ turns the growth constraint into a constraint on the __excess__ growth rate:
$$
\begin{align*}
\left(\hat{\mathbf{g}}_{\text{SIM}}-g_{f}\mathbf{1}\right)^{\top}\mathbf{w}&\geq g_{\star}-g_{f}
\end{align*}
$$
which has a consequence worth seeing. Scaling every risky weight by a positive constant scales the excess growth by that constant and the variance by its square, and, as long as no upper bound $w_{i}\leq1$ is active, the constraint set $w_{i}\geq0$ is unchanged. So, for feasible targets $g_{\star}>g_{f}$ and a positive definite $\hat{\mathbf{\Sigma}}_{g,\text{SIM}}$ (which makes the minimizer unique), the minimizers are __one__ risky direction scaled up or down: the solutions lie on a ray from the risk-free asset. Writing $\mathbf{w}_{\mathcal{T}}$ for the point on the ray with $\mathbf{1}^{\top}\mathbf{w}=1$ (fully invested in risky assets, $w_{f}=0$), the solution for a target $g_{\star}$ is:
$$
\boxed{
\begin{align*}
\mathbf{w}(g_{\star}) &= (1-w_{f})\,\mathbf{w}_{\mathcal{T}},\qquad 1-w_{f} = \frac{g_{\star}-g_{f}}{\mathbb{E}[g_{\mathcal{T}}]-g_{f}}
\end{align*}}
$$
valid until the first bound binds, at $(1-w_{f})\max_{i}w_{\mathcal{T},i}=1$, beyond which the boundary kinks. Targets below $\mathbb{E}[g_{\mathcal{T}}]$ lend ($0<w_{f}<1$), targets above it borrow ($w_{f}<0$), and $\mathbf{w}_{\mathcal{T}}$ itself is recovered from __any__ solution on the ray by normalizing, $\mathbf{w}_{\mathcal{T}}=\mathbf{w}/\mathbf{1}^{\top}\mathbf{w}$ (for $g_{\star}=g_{f}$ the solution is $\mathbf{w}=\mathbf{0}$ and there is nothing to normalize). The second example sweeps the target, checks that every normalized solution is the same vector, and reads $\mathbf{w}_{\mathcal{T}}$ off the ray. That fully invested portfolio is the tangent portfolio of L5b, as the next section recalls, and the figure sketches the whole picture in the risk-growth plane (portfolio 1 is the tangent portfolio, portfolio 2 the global minimum-variance portfolio, and the dashed lower branch is dominated).

<div>
    <center>
        <img src="figs/Fig-MinVar-RiskFree-Schematic.svg" width="640" alt="Schematic of the capital allocation line: the risky efficient frontier as a curve opening to the right with the GMV portfolio at its vertex, the risk-free asset at (0, g_f) on the vertical axis, and a straight line from the risk-free asset tangent to the efficient frontier at the tangent portfolio; the expected growth rates and growth-rate standard deviations of the tangent and GMV portfolios are marked on the axes"/>
    </center>
</div>

### The capital allocation line and the tangent portfolio (recall from L5b)
Take any risky portfolio $p$ with expected growth rate $\mathbb{E}[g_{p}]$ and growth-rate standard deviation $\sigma_{g,p}>0$, and put a fraction $w_{f}$ of wealth in the risk-free asset and $1-w_{f}$ in $p$. The __complete portfolio__ $c$ has $\mathbb{E}[g_{c}]=g_{f}+(1-w_{f})(\mathbb{E}[g_{p}]-g_{f})$ and $\sigma_{g,c}=|1-w_{f}|\,\sigma_{g,p}$, and eliminating $w_{f}$ (for $w_{f}\leq1$) gives the __capital allocation line__ (CAL) through $p$:
$$
\begin{align*}
\mathbb{E}\left[g_{c}\right] &= g_{f} + \underbrace{\left(\frac{\mathbb{E}\left[g_{p}\right]-g_{f}}{\sigma_{g,p}}\right)}_{\text{Sharpe ratio}\;\text{SR}_{p}}\;\sigma_{g,c}
\end{align*}
$$
a straight line from $(0,g_{f})$ through $p$ in the $(\sigma_{g},g)$ plane whose slope is the __Sharpe ratio__ of $p$. A steeper line is better at every level of risk, so the risky portfolio to hold is the one with the largest Sharpe ratio, the __tangent portfolio__ $\mathcal{T}$, whose CAL touches the efficient frontier at $\mathcal{T}$. The ray of the previous section is that CAL: the excess-growth constraint asks for the least variance per unit of excess growth, which is the largest Sharpe ratio, so the ray's direction is the (long-only) maximum-Sharpe portfolio, and the example confirms it against the maximum-Sharpe point of the risky-only frontier. Under the SIM inputs the tangent portfolio's Sharpe ratio reads $\left(\hat{\boldsymbol{\alpha}}^{\top}\mathbf{w}_{\mathcal{T}}+\hat{\beta}_{\mathcal{T}}\,g^{\prime}_{M}-g_{f}\right)\big/\sqrt{s^{2}_{g,M}\hat{\beta}_{\mathcal{T}}^{2}+\mathbf{w}_{\mathcal{T}}^{\top}\hat{\mathbf{D}}_{g}\mathbf{w}_{\mathcal{T}}}$ with $\hat{\beta}_{\mathcal{T}}=\hat{\boldsymbol{\beta}}^{\top}\mathbf{w}_{\mathcal{T}}$ the tangent portfolio's beta.

Two things carry over from L5b unchanged. When short positions are allowed, $\hat{\mathbf{\Sigma}}_{g,\text{SIM}}$ is positive definite, and the GMV portfolio earns more than $g_{f}$ in expectation, the tangent portfolio has the closed form $\mathbf{w}_{\mathcal{T}}\propto\hat{\mathbf{\Sigma}}_{g,\text{SIM}}^{-1}(\hat{\mathbf{g}}_{\text{SIM}}-g_{f}\mathbf{1})$ normalized to sum to one (the rank-one-plus-diagonal structure even gives that inverse in closed form; see the notes), while under the long-only constraint there is no closed form and the ray or the frontier sweep is how it is found. And the Sharpe ratio above, with $\sigma_{g,p}$ in the denominator, is the CAL slope in growth-rate units, the Sharpe ratio of a one-observation-interval holding period; the convention in practice divides by the volatility $\sqrt{\Delta{t}}\,\sigma_{g,p}$ instead, with $\Delta{t}$ the observation interval in years (one trading day here), giving the __annualized__ Sharpe ratio $\text{SR}_{p}/\sqrt{\Delta{t}}$, larger by $\sqrt{252}\approx15.9$ for daily data. The two rank portfolios identically; from here on the course quotes the annualized value, and the example finds about $1.3$ for both the SIM and the data-driven tangent portfolios on the training data.

### Separation, and what breaks it
Because the tangent portfolio's CAL dominates every other, all mean-variance investors who share the same estimates, the same $g_{f}$, the same horizon, and the same opportunity set (universe and constraints) in a frictionless market hold the same risky fund $\mathcal{T}$ and differ only in $w_{f}$: the investment decision is separated from the financing decision, Tobin's __two-fund separation__ (L5b). The result is exact for the unconstrained problem with one lending and borrowing rate, and it survives the long-only constraint on the risky weights alone, which is why the package's ray exists: the risky fund becomes the constrained maximum-Sharpe portfolio and investors still differ only in $w_{f}$. What breaks it is a constraint on the risk-free leg or on dollar positions:

* __No borrowing__ ($w_{f}\geq0$) truncates the straight CAL segment at $\mathcal{T}$; above it, the efficient boundary generally returns to the risky-only frontier, so investors who want more growth than $\mathbb{E}[g_{\mathcal{T}}]$ hold different risky portfolios, and separation holds only piecewise. The package does not impose $w_{f}\geq0$; adding it changes the answer above $\mathcal{T}$.
* __Position bounds__ ($w_{i}\leq1$ here, or a concentration limit $w_{i}\leq u$) kink the ray where they bind. Practitioners often impose concentration limits because unconstrained maximum-Sharpe portfolios put extreme weights on a few assets when the mean growth rates are imprecise; the limits lower the in-sample Sharpe ratio and may make the portfolio more robust to estimation error, which is a claim to test, not to assume.
* __A borrowing rate above the lending rate__ kinks the line at $\mathcal{T}$: the lending segment keeps its slope and the borrowing extension is flatter.

Let's look at the example that builds the ray, reads off the tangent portfolio, and scores complete portfolios that lend, hold, or borrow.

> __Example__
>
> [▶ SIM portfolios with a risk-free asset, the tangent portfolio, and the capital allocation line](CHEME-5660-L6b-Example-SIM-MinVar-RRFA-Fall-2026.ipynb). Solve the risky and risk-free problem with SIM inputs as the package poses it, show that every solution lies on one ray from the risk-free asset, read the tangent portfolio off the ray and check it against the maximum-Sharpe point of the risky-only frontier, compare it with the data-driven tangent portfolio, and score complete portfolios that lend, hold, or borrow on 2025 data.

___

## What the SIM Inputs Change, and How to Check
Put the pieces together. The SIM does not change the reward input at all, it replaces the sample covariance by a rank-one-plus-diagonal matrix that omits the residual covariances, and it does so with $2|\mathcal{P}|+1$ estimated numbers instead of $|\mathcal{P}|(|\mathcal{P}|+1)/2$. For a universe like the example's, where most residual correlations are small (median absolute value $0.14$) even though a few industry pairs reach $0.6$ to $0.7$, that costs little in sample: the SIM weights select nearly the same assets as the data-driven weights, differ by up to seven percentage points, and their growth-rate standard deviation under the data covariance is at most a tenth of a unit above the data-optimal value. The cost grows with the residual correlations among the assets the optimizer actually holds; two banks or two share classes of one firm are where it bites (L6a).

What the SIM buys is not visible in a thirteen-firm example, and it is worth saying plainly: with hundreds of assets and a few years of daily data, the sample covariance has more parameters than the data can pin down and its inverse amplifies the noisiest directions (L5b's advanced material), while the SIM covariance is estimated from a few numbers per asset, is always positive semidefinite, and its structure is interpretable. Whether the parsimony helps out of sample is an empirical question about the universe.

Checking is where discipline matters, and the checks are those of L5b. Fit everything on the training window and freeze it: the universe, the SIM parameters, the market moments, $g_{f}$ at the decision date, and the weights. Then evaluate the frozen weights with the buy-and-hold wealth rule on the test year, next to simple baselines (equal weights, the GMV portfolio, an index fund). Read the result for what it is: one year, one draw, one universe. In the examples the paired SIM and data-driven portfolios earned nearly the same 2025 wealth, which shows that the two input sets selected nearly the same portfolios, and which cannot rank the two input sets, because the gap between them is far smaller than the sampling error in either. The realized risk of the complete portfolios scaled almost exactly with the risky fraction, as the CAL predicts for the linearized model (not exactly: a buy-and-hold mixture drifts during the year); the realized growth did not match the model, because it depends on whether the tangent portfolio's estimated growth rate persisted, the input least likely to.

Finally, the parameters in the archive are estimates with the standard errors L6a computed, and a portfolio built from them inherits that uncertainty in a nonlinear way: narrow intervals on every beta do not imply stable weights. Propagating the estimation uncertainty through the covariance into the weights and the portfolio variance is the subject of the optional advanced notebook below, and it is the question to ask before any SIM portfolio is traded.
___

## Optional Advanced Material
The notebook below extends today's material. It is optional and is not a prerequisite for L7b; the [advanced index](advanced/README.md) describes it.

* [▶ Propagating SIM parameter uncertainty into a portfolio](advanced/uncertainty/CHEME-5660-L6b-Advanced-SIM-Portfolio-Uncertainty-Fall-2026.ipynb). Bootstrap the SIM parameters of a small universe two ways (empirical residuals and Gaussian innovations), keep each asset's joint draw of intercept, beta, and residual scale, rebuild the SIM covariance for every draw, and read the resulting distributions of portfolio volatility, allocation distance, and variance regret for a fixed unconstrained minimum-variance allocation.
___

## Summary
In this lecture, we assembled the single index model inputs for a portfolio, showed exactly how they relate to the data-driven inputs, solved the risky-only and the risky and risk-free allocation problems with them, recalled the capital allocation line, the tangent portfolio, and two-fund separation in the form the package computes, and compared the SIM and data-driven portfolios in sample and out of sample.

> __Key Takeaways:__
>
> * **The SIM changes only the covariance, and we know how:** Fitted with an intercept on the same data, the SIM mean vector equals the sample-mean vector, and the SIM covariance equals the sample covariance with the off-diagonal residual covariances removed (and a negligible degrees-of-freedom factor on the diagonal), so every difference between the SIM and data-driven portfolios traces to a residual covariance the model set to zero, understating risk where residuals are positively correlated and overstating it where they are negatively correlated.
> * **The risky and risk-free problem has one risky answer:** The package leaves the risk-free fraction free, so the growth constraint is a constraint on excess growth and every target selects the same risky direction scaled by $1-w_{f}$; the fully invested point on that ray is the long-only tangent portfolio, the ray is its capital allocation line, and a borrowing restriction, position bounds, or a higher borrowing rate are what kink or truncate it.
> * **Compare in sample by regret and out of sample by frozen weights:** The SIM weights evaluated under the data covariance can only lose to the data-optimal weights at the same target, and by little for a universe whose residual correlations are mostly small; on the test year the paired portfolios were nearly indistinguishable, which is a check on the weights, not a ranking of the input sets, and the parameter uncertainty of L6a still has to be propagated into the weights before either is trusted.

Next time, in L7b, we let the weights evolve through time and distinguish a rebalancing policy from the one-time allocation of this lecture.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___